In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from tqdm import trange
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
dataset_path = "/opt/gpudata/ecg/code15"
_path = Path(dataset_path)

### Create Initial Train/Val/Test

Refer to https://zenodo.org/records/4916206 for source dataset details

* binarize survival labels
* save indices within h5

In [ ]:
src = pd.read_csv(_path / "exams.csv")
n_shards = 18

In [ ]:
meta = []
for i in trange(n_shards):
    with h5py.File(_path / f"exams_part{i}.hdf5", "r") as h5:
        exam_ids = h5["exam_id"][:]
        # X = h5["tracings"][:]
        for j, exam_id in enumerate(exam_ids):
            meta.append(
                {
                    "exam_id": exam_id,
                    "shard_num": i,
                    "shard_idx": j, # index within the shard
                }
            )
meta = pd.DataFrame(meta)
print(f"found {len(meta)} records in h5 files")
# seems like there's some padded records (exam_id=0, signal entirely 0s) at the end of each shard
meta = meta.drop_duplicates("exam_id", keep=False).reset_index(drop=True)
print(f"filtered to {len(meta)} records after removing duplicate exam IDs")

In [ ]:
assert meta["exam_id"].is_unique
assert src["exam_id"].is_unique
df = meta.merge(src, on="exam_id")

In [ ]:
# survival outcomes only defined on first ecg per patient
first_ecg_mask = df["timey"].notna()
df = df[first_ecg_mask].reset_index(drop=True)
assert df["patient_id"].is_unique
assert df["death"].notna().all()

In [ ]:
# binarize survival to mortality at T years
# allow any observed death
# for non-observed death, require last followup to be after T years to ensure T year survival
threshold = src["timey"].median()

df = df[
    df["death"] |
    ((~df["death"]) & (df["timey"] >= threshold))
].reset_index(drop=True)

df["mortality"] = df["death"] & (df["timey"] < threshold)

labels = ["1dAVb", "RBBB", "LBBB", "SB", "ST", "AF", "mortality"]

In [ ]:
# stratify by age/sex/mort, roughly makes the other labels equivalent
binned_age = pd.cut(df["age"], [0, 20, 40, 60, 80, 300]).cat.codes
sex = df["is_male"].astype(int)
mort = df["mortality"].astype(int)
stratifier = binned_age.astype(str) + "_" + sex.astype(str) + mort.astype(str)

In [ ]:
train_df, _vt_df, _, _vt_strat = train_test_split(df, stratifier, test_size=0.4, random_state=42, stratify=stratifier)
val_df, test_df = train_test_split(_vt_df, test_size=0.5, random_state=42, stratify=_vt_strat)

In [ ]:
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

save_cols = ["patient_id", "exam_id", "shard_num", "shard_idx", "age", "is_male"] + labels + ["split"]
save_df = pd.concat([train_df, val_df, test_df], ignore_index=True)[save_cols]
save_df[labels] = save_df[labels].astype(int)
save_df

In [ ]:
save_df["split"].value_counts()

In [ ]:
save_df.to_csv(_path / "labels.csv", index=False)

### Compute Waveform Stats

derive train split waveform stats for normalization

In [ ]:
df = pd.read_csv(_path / "labels.csv")
train_df = df[df["split"] == "train"].reset_index(drop=True)

In [ ]:
# gather waveforms from all shards
all_waveforms = []
shard_lens = []
n_shards = 18
for i in trange(n_shards):
    with h5py.File(_path / f"exams_part{i}.hdf5", "r") as h5:
        X = h5["tracings"][:]
    all_waveforms.append(X)
    shard_lens.append(X.shape[0])
all_waveforms = np.concatenate(all_waveforms)
shard_starts = np.cumsum(shard_lens) - shard_lens[0]
shard_to_offset = {i: s for i, s in enumerate(shard_starts)}

In [ ]:
# convert shard indices into global index
train_df["global_idx"] = train_df["shard_num"].replace(shard_to_offset) + train_df["shard_idx"]
assert train_df["global_idx"].is_unique

train_waveforms = all_waveforms[train_df["global_idx"].to_numpy()]
train_waveforms.shape

In [ ]:
# supposedly, all waveforms are zero padded
# per the authors, recordings were 10 or 7 seconds, both at 400 Hz
# for simplicity, just use padded waveforms to compute stats
lowers, uppers = np.percentile(train_waveforms, [0.1, 99.9], axis=(0, 1))
display(lowers.tolist())
display(uppers.tolist())

train_waveforms_clipped = np.clip(train_waveforms, lowers, uppers)
means = train_waveforms_clipped.mean(axis=(0, 1))
stds = train_waveforms_clipped.std(axis=(0, 1))
display(means.tolist())
display(stds.tolist())

### Create Nested Train Subsets

just stratify by age/sex stratification

In [ ]:
# TODO